# Stage 2 Notebook 35 - Exp2DD Self-distillation from Exp2Z teacher (project namesake KD)

**Why this exists.** The Exp2Z oracle diagnostic showed `decoded_f1=0.044` vs `oracle_f1=0.101` -- the cls head throws away 56% of available geometric quality. No supervision change has broken this gap.

Knowledge distillation is the project's literal name (CLRKD = CLRKDNet KD) and has never been implemented. The plumbing already exists in `train_joint_model_experiment.py` (`load_lane_teacher`) and `losses.py` (`w_distill` field) but `w_distill: 0.0` everywhere. We've never turned it on.

Exp2DD self-distills from Exp2Z's best.pt:
- Teacher = frozen Exp2Z lane_head, runs forward on every batch (no_grad).
- Soft targets: teacher's `cls_logits` and `coord_pred` are detached and used as auxiliary regression targets for the student.
- KD loss: `w_distill * (mse(student_cls, teacher_cls) + masked_l1(student_curves, teacher_curves))`.
- Student also has the normal hard-target supervision (matched-existence, IoU regression, etc.).

**Critical setup step before running**: the config `teacher.lane_head_checkpoint` must point to the .pt file from Exp2Z (NB31). On Colab after extracting the tar:

```
/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_best.pt
```

If the .pt isn't extracted from the tar yet, run the cell below this markdown to extract it.

Reference: This is the simplest form of self-distillation (matching teacher predictions). More sophisticated variants (FitNet feature distillation, attention transfer, RKD relation-based) are possible follow-ups.

### Run mode

1. **First**: extract teacher checkpoint from Exp2Z tar (cell below).
2. Keep `DEBUG_MODE = True` for the first run.
3. After smoke + debug pass, change to `False` for the 15-epoch short run.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

# --- Extract teacher checkpoint from Exp2Z tar (one-time setup) ---
import tarfile, shutil
TEACHER_TAR = '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar'
TEACHER_DEST = '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_best.pt'

if not os.path.exists(TEACHER_DEST):
    if not os.path.exists(TEACHER_TAR):
        raise FileNotFoundError(f'Teacher tar not found: {TEACHER_TAR}. Run NB31 (Exp2Z) first.')
    print(f'Extracting best.pt from teacher tar: {TEACHER_TAR}', flush=True)
    with tarfile.open(TEACHER_TAR, 'r') as tar:
        for member in tar.getmembers():
            if member.name.endswith('best.pt'):
                with tar.extractfile(member) as src, open(TEACHER_DEST, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                print(f'Extracted to: {TEACHER_DEST}', flush=True)
                break
        else:
            raise FileNotFoundError('best.pt not found in teacher tar.')
else:
    print(f'Teacher checkpoint already at: {TEACHER_DEST}', flush=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane
Extracting best.pt from teacher tar: /content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar
Extracted to: /content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_best.pt


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp30_rmt_gca_mask_self_distillation_joint_smoke.log
OK exp30_rmt_gca_mask_self_distillation_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=2.9634 det_loss=3.5104 grad_cos=-0.1833 lambda_lane=0.0980
  gate_stats={'gate/det_mean': 0.5012585520744324, 'gate/lane_mean': 0.4999748766422272, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short15'
    EPOCHS = 15
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp30_rmt_gca_mask_self_distillation_joint_short15 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_short15.tar --epochs 15 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_short15.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp30_rmt_gca_mask_self_distillation_joint_short15_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml --curve-tar /content/d

CalledProcessError: Command '['/usr/bin/python3', '-u', 'stage2/scripts/train_joint_model_experiment.py', '--config', 'stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml', '--curve-tar', '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar', '--curve-root', '/content/bdd100k_clrkd_curve', '--work-dir', '/content/exp30_rmt_gca_mask_self_distillation_joint_short15', '--output-tar', '/content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_short15.tar', '--epochs', '15', '--batch-size', '8', '--limit-train', '3000', '--limit-val', '1000', '--force-extract', '--print-every', '5']' returned non-zero exit status 1.

## What to watch in Exp2DD training

Pass criteria at epoch 15:
- **`val/lane/decoded_f1 >= 0.07`**: KD lifts cls toward oracle. The smoking gun for KD's value.
- **`val/lane/decoded_oracle_f1`** stays >= Exp2Z's 0.101 (geometry doesn't regress).
- **`val/lane/distill`** loss component appears in epoch summary, decreasing over training.
- **`val/lane_exist_best_f1 >= 0.70`**: cls separation improves with teacher signal.

Failure signals:
- decoded_f1 stays at ~0.044: teacher's signal is no better than the loss function's; we've reached the architecture's true capacity.
- Teacher loading errors at startup: check that NB31 (Exp2Z) finished and produced exp26_*.tar in Drive.